# MinerU PP-DocLayoutV2 — real IBM ART vision attacks

This notebook now targets a concrete, actual PyTorch model file inside the official MinerU model bundle: `Layout/PP-DocLayoutV2/model.safetensors`. It loads the Hugging Face Transformers PP-DocLayoutV2 architecture from the local model directory, performs a real forward pass, wraps its actual object-query logits with IBM ART's `PyTorchClassifier`, and runs bounded image-evasion attacks.

**Why this model first?** It is the clearest image-input PyTorch model in the bundle with a supported Transformers architecture and differentiable output. The OCR folder contains separate detection/recognition weights, and the VLM is a separate GGUF/mmproj model. They are different components, not interchangeable weights. This notebook focuses on layout detection only, not OCR recognition, VLM reasoning, or end-to-end Markdown output.

The ART adapter turns per-query layout logits into per-class presence scores (`max` over queries) so classifier-style ART attacks can operate. Interpret results as robustness of layout-class presence, not a complete attack on bounding-box geometry. Run in Colab with GPU recommended. It creates an HTML report and log.

In [ ]:
# Real IBM ART attacks against MinerU's actual PP-DocLayoutV2 layout model
from pathlib import Path
import os, sys, json, time, html, traceback, hashlib, subprocess
from datetime import datetime, timezone
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from safetensors.torch import load_file
from transformers import PPDocLayoutV2Config, PPDocLayoutV2ForObjectDetection
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import (
    FastGradientMethod, ProjectedGradientDescent, BasicIterativeMethod,
    SquareAttack, ZooAttack, NewtonFool, SpatialTransformation,
    PixelAttack, ThresholdAttack
)

ROOT=Path("/content/mineru_ibm_art_layout_attacks")
MODELS=Path.home()/".mineru"/"models"/"MinerU-4_models_torch"/"Layout"/"PP-DocLayoutV2"
OUT=ROOT/"artifacts"; IN=ROOT/"inputs"
for d in (OUT,IN): d.mkdir(parents=True,exist_ok=True)
LOG=OUT/"mineru_ibm_art_layout.log"; REPORT=OUT/"mineru_ibm_art_layout.html"; lines=[]
def log(s):
 print(s); lines.append(str(s))
def esc(s): return html.escape(str(s),quote=True)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"UTC: {datetime.now(timezone.utc).isoformat()}")
log(f"Python: {sys.version}")
log(f"PyTorch: {torch.__version__}; device={device}")
log(f"Model directory: {MODELS}")

# Ensure the MinerU model bundle exists. Current MinerU official model manager is used.
if not (MODELS/"model.safetensors").exists():
    log("MinerU layout weights not found; downloading official MinerU Torch bundle.")
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","mineru[torch]>=4.0,<5","adversarial-robustness-toolbox"],check=True)
    subprocess.run(["mineru-kit","models","download","--tier","basic","--small-backend","torch","--source","huggingface"],check=True)
else:
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","adversarial-robustness-toolbox"],check=True)
assert (MODELS/"model.safetensors").exists(), f"Missing real model weights at {MODELS}"
assert (MODELS/"config.json").exists(), f"Missing model config at {MODELS}"
log("Weight file size: "+str((MODELS/"model.safetensors").stat().st_size))
config=PPDocLayoutV2Config.from_pretrained(str(MODELS))
base_model=PPDocLayoutV2ForObjectDetection.from_pretrained(str(MODELS),config=config,torch_dtype=torch.float32)
base_model.to(device).eval()
log(f"MODEL LOADED: {base_model.__class__.__name__}; params={sum(p.numel() for p in base_model.parameters()):,}")

# Use the actual PP-DocLayoutV2 object-detection network. ART's classifier interface is
# adapted to a documented proxy task: per-class presence scores are max logits over object queries.
# This tests robustness of layout class detection, not the full end-to-end Markdown output.
class LayoutPresence(torch.nn.Module):
    def __init__(self, model):
        super().__init__(); self.model=model
    def forward(self, x):
        # ART inputs are float RGB [0,1], BCHW. Match MinerU's preprocessing: resize then /255.
        x=torch.nn.functional.interpolate(x, size=(800,800), mode="bicubic", align_corners=False)
        x=x*255.0
        out=self.model(pixel_values=x)
        # shape [batch, queries, classes]; class presence proxy = max object-query logit
        return out.logits.max(dim=1).values
wrapped=LayoutPresence(base_model).to(device).eval()
# smoke test real forward pass
dummy=torch.rand(1,3,320,320,device=device)
with torch.no_grad(): raw=wrapped(dummy)
assert raw.ndim==2 and raw.shape[1]>1, f"Unexpected output shape: {tuple(raw.shape)}"
nclasses=raw.shape[1]
log(f"FORWARD PASS: input={tuple(dummy.shape)}, output={tuple(raw.shape)}, classes={nclasses}")
criterion=torch.nn.CrossEntropyLoss()
classifier=PyTorchClassifier(model=wrapped,loss=criterion,input_shape=(3,320,320),
    nb_classes=nclasses,optimizer=torch.optim.Adam(wrapped.parameters(),lr=1e-5),
    clip_values=(0.0,1.0),device_type="gpu" if device.type=="cuda" else "cpu")

# Generate one reproducible document-like test image (image values normalized [0,1]).
img=Image.new("RGB",(900,1200),"white"); dr=ImageDraw.Draw(img)
try: font=ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",34)
except: font=ImageFont.load_default()
for y,t in [(70,"PURCHASE ORDER"),(190,"Order Number: PO-2026-1042"),(280,"Supplier: Example Components Ltd."),(390,"Total: 1250.00 USD"),(500,"Status: APPROVED"),(700,"Terms and Conditions"),(790,"Delivery within 30 days.")]:
    dr.text((65,y),t,fill="black",font=font)
image_path=IN/"layout_test_document.png"; img.save(image_path)
arr=np.asarray(img.resize((320,320)),dtype=np.float32)/255.0
x=np.transpose(arr,(2,0,1))[None,...].astype(np.float32)
y_pred=classifier.predict(x)
y=int(np.argmax(y_pred[0]))
log(f"Baseline predicted class index={y}; score={float(np.max(y_pred[0])):.6f}")

# All selected attacks are real ART implementations. Parameters intentionally bounded for Colab.
# Some query attacks can be slow; use a per-attack time limit and record honest failures.
attack_specs=[
 ("FastGradientMethod",lambda:FastGradientMethod(estimator=classifier,eps=8/255)),
 ("ProjectedGradientDescent",lambda:ProjectedGradientDescent(estimator=classifier,eps=8/255,eps_step=2/255,max_iter=5)),
 ("BasicIterativeMethod",lambda:BasicIterativeMethod(estimator=classifier,eps=8/255,eps_step=2/255,max_iter=5)),
 ("SquareAttack",lambda:SquareAttack(estimator=classifier,max_iter=20,eps=8/255,p_init=0.1)),
 ("ZooAttack",lambda:ZooAttack(classifier=classifier,confidence=0.0,targeted=False,learning_rate=1e-2,max_iter=10,binary_search_steps=1,initial_const=1e-2,abort_early=True,use_resize=False,nb_parallel=8,batch_size=1)),
 ("NewtonFool",lambda:NewtonFool(classifier=classifier,max_iter=5,eta=0.01)),
 ("SpatialTransformation",lambda:SpatialTransformation(estimator=classifier, max_translation=2, num_translations=3, max_rotation=5, num_rotations=3)),
 ("PixelAttack",lambda:PixelAttack(estimator=classifier,max_iter=10,th=10)),
 ("ThresholdAttack",lambda:ThresholdAttack(estimator=classifier,threshold=0.1)),
]
results=[]
for name,make in attack_specs:
    start=time.time(); row={"attack":name,"status":"NOT_RUN","baseline_class":y}
    try:
        atk=make()
        adv=atk.generate(x=x,y=np.array([y],dtype=np.int64))
        pred=classifier.predict(adv)
        new=int(np.argmax(pred[0]))
        row.update({"status":"EXECUTED","adversarial_class":new,"changed_top_class":new!=y,
                    "baseline_score":float(np.max(y_pred[0])),"adversarial_score":float(np.max(pred[0])),
                    "linf":float(np.max(np.abs(adv-x))),"duration_s":round(time.time()-start,2)})
        row["interpretation"]="Potential layout-class change" if new!=y else "No top-class change in this run"
        log(json.dumps(row))
    except Exception as e:
        row.update({"status":"FAILED","error":repr(e),"traceback":traceback.format_exc(),"duration_s":round(time.time()-start,2)})
        log(json.dumps(row))
    results.append(row)

# Save adversarial evidence by attack; this is input perturbation against layout class presence proxy.
summary={"utc":datetime.now(timezone.utc).isoformat(),"model":"MinerU PP-DocLayoutV2 (actual model.safetensors)",
 "model_path":str(MODELS),"model_loaded":True,"forward_pass":True,"device":str(device),
 "attack_count":len(results),"executed":sum(r["status"]=="EXECUTED" for r in results),
 "failed":sum(r["status"]=="FAILED" for r in results),
 "class_proxy":"max object-query logits per layout class; not full document parse",
 "baseline_class_index":y,"limitations":"These attacks target the actual PP-DocLayoutV2 layout detector through an ART classifier adapter. They do not directly attack the separate OCR safetensors, GGUF VLM, or end-to-end MinerU Markdown pipeline."}
log("SUMMARY "+json.dumps(summary,indent=2))
rows="".join("<tr>"+"".join(f"<td>{esc(r.get(k,''))}</td>" for k in ["attack","status","baseline_class","adversarial_class","changed_top_class","linf","duration_s","interpretation","error"])+"</tr>" for r in results)
report=f"""<!doctype html><html><head><meta charset="utf-8"><title>MinerU PP-DocLayoutV2 IBM ART Report</title><style>body{{font:14px/1.5 Arial;margin:0;color:#172033}}header{{background:#172b4d;color:white;padding:24px}}main{{max-width:1300px;margin:24px auto;padding:0 18px}}.note{{background:#eef6ff;border-left:4px solid #2766a8;padding:14px;margin:18px 0}}table{{border-collapse:collapse;width:100%;font-size:12px}}th,td{{border:1px solid #d7dee8;padding:8px;vertical-align:top;overflow-wrap:anywhere}}th{{background:#eaf0f7}}pre{{white-space:pre-wrap;overflow-wrap:anywhere;background:#f6f8fb;padding:12px}}</style></head><body><header><h1>MinerU PP-DocLayoutV2 — IBM ART Image Attack Report</h1><div>Actual local safetensors model · PyTorch forward pass · ART attacks</div><small>{esc(summary['utc'])}</small></header><main><div class="note"><b>Scope:</b> ART attacks run against the actual MinerU PP-DocLayoutV2 layout detector. The ART adapter maps its object-query logits to per-class presence scores using max over queries. This makes classifier attacks usable, but evaluates layout-class presence—not the full detection box geometry or final Markdown output. OCR and VLM are separate components.</div><h2>Run summary</h2><pre>{esc(json.dumps(summary,indent=2))}</pre><h2>Attack results</h2><table><thead><tr>{"".join(f"<th>{esc(k)}</th>" for k in ["attack","status","baseline_class","adversarial_class","changed_top_class","linf","duration_s","interpretation","error"])}</tr></thead><tbody>{rows}</tbody></table><h2>Environment / evidence</h2><pre>Model: {esc(MODELS)}\nSHA-256: {hashlib.sha256((MODELS/'model.safetensors').read_bytes()).hexdigest()}\nInput image: {esc(image_path)}\nInput SHA-256: {hashlib.sha256(image_path.read_bytes()).hexdigest()}\nDevice: {esc(device)}\nRaw baseline logits: {esc(y_pred.tolist())}</pre><h2>Full results JSON</h2><pre>{esc(json.dumps(results,indent=2))}</pre></main></body></html>"""
REPORT.write_text(report,encoding="utf-8"); LOG.write_text("\n".join(lines)+"\n",encoding="utf-8")
print("HTML report:",REPORT); print("Log:",LOG)


## Download the generated report and log

In [ ]:
from pathlib import Path
from google.colab import files
for path in [Path("/content/mineru_ibm_art_layout_attacks/artifacts/mineru_ibm_art_layout.html"),
             Path("/content/mineru_ibm_art_layout_attacks/artifacts/mineru_ibm_art_layout.log")]:
    print(path, "exists=", path.exists(), "bytes=", path.stat().st_size if path.exists() else 0)
    if path.exists(): files.download(str(path))
